# Synthetic end-to-end validation — L1 abundance recovery (5.02)The classifier-filter → salmon pipeline vs the baseline methods on the synthetic chr1L1-expression benchmark (model 2: standalone full-length L1 transcripts, per-UID copycount = ground truth).**Locus resolution is a sequence limit, not a pipeline failure.** Young full-length L1are >99% identical, so short reads cannot be assigned to an individual locus. Recoveryimproves with aggregation, so R² is reported per-locus → total for every method.- Inputs: `results/synthetic_validation/abundance.csv` (seqlabel) and  `abundance_methods.csv` (all methods) — from `scripts/python/synthetic_abundance.py`- Figures → `reports/figures/synthetic_validation/`; R² tables → `results/synthetic_validation/`

In [ ]:
library(data.table)library(ggplot2)OKABE_ITO <- c("#E69F00", "#56B4E9", "#009E73", "#F0E442",               "#0072B2", "#D55E00", "#CC79A7", "#000000")theme_set(theme_bw(base_size = 14))results_dir <- file.path("..", "results", "synthetic_validation")figures_dir <- file.path("..", "reports", "figures", "synthetic_validation")dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)save_fig <- function(plot, stem, width, height) {  for (ext in c("png", "pdf")) {    suppressMessages(ggsave(file.path(figures_dir, paste0(stem, ".", ext)),                            plot, width = width, height = height, dpi = 300, bg = "white"))  }  invisible(plot)}r2 <- function(x, y) {  if (length(x) < 3 || sd(x) == 0 || sd(y) == 0) return(NA_real_)  summary(lm(y ~ x))$r.squared}

In [ ]:
# --- seqlabel table (aggregation-ladder analysis) --------------------------ab <- fread(file.path(results_dir, "abundance.csv"))ab[, power := as.integer(power)]cat(sprintf("seqlabel: rows=%d  cells=%d  elements=%d\n",            nrow(ab), uniqueN(ab[, .(power, del_prob)]), uniqueN(ab$l1_id)))head(ab)

In [ ]:
# --- R^2 at three aggregation levels (AlbertSalmon_seqlabel) ---------------ab_sub <- ab[, .(sim = sum(simulated), est = sum(albertsalmon_seqlabel)),             by = .(power, del_prob, subfamily)]ab_tot <- ab[, .(sim = sum(simulated), est = sum(albertsalmon_seqlabel)),             by = .(power, del_prob)]overall <- data.table(  level = factor(c("per-locus", "per-subfamily", "total (family)"),                 levels = c("per-locus", "per-subfamily", "total (family)")),  r2 = c(r2(ab$simulated, ab$albertsalmon_seqlabel), r2(ab_sub$sim, ab_sub$est), r2(ab_tot$sim, ab_tot$est)))fwrite(overall, file.path(results_dir, "r2_by_aggregation.csv"))print(overall)

In [ ]:
# --- Figure: synthetic_albertsalmon_by_level (the aggregation ladder) ------fig_level <- ggplot(overall, aes(level, r2, fill = level)) +  geom_col(width = 0.62) +  geom_text(aes(label = sprintf("%.2f", r2)), vjust = -0.4, size = 4.2) +  scale_fill_manual(values = OKABE_ITO[c(6, 1, 3)], guide = "none") +  coord_cartesian(ylim = c(0, 1.08)) +  labs(x = "Aggregation level", y = expression(R^2 ~ "(recovered vs simulated)"))save_fig(fig_level, "synthetic_albertsalmon_by_level", 6, 4)fig_level

In [ ]:
# --- Figure: synthetic_albertsalmon_scatter (total / family level) ---------r2_tot <- overall[level == "total (family)", r2]fig_scatter <- ggplot(ab_tot, aes(sim, est)) +  geom_point(size = 2.6, alpha = 0.75, colour = OKABE_ITO[2]) +  geom_smooth(method = "lm", se = TRUE, colour = OKABE_ITO[6], fill = OKABE_ITO[6]) +  annotate("text", x = -Inf, y = Inf, hjust = -0.15, vjust = 1.6,           label = sprintf("R^2 == %.3f", r2_tot), parse = TRUE, size = 5) +  labs(x = "Simulated total L1 expression (transcript copies)",       y = "Recovered total (salmon NumReads)")save_fig(fig_scatter, "synthetic_albertsalmon_scatter", 5, 5)fig_scatter

In [ ]:
# --- multi-method comparison (all baselines that have run) -----------------# abundance_methods.csv: power, del_prob, method, l1_id, simulated, abundance.methods_file <- file.path(results_dir, "abundance_methods.csv")have_methods <- file.exists(methods_file)if (have_methods) {  M <- fread(methods_file)  M[, power := as.integer(power)]  # each method has its own units → scale to the copy-count scale (R^2 is unaffected).  M[, scale := sum(simulated) / sum(abundance), by = method]  M[is.finite(scale) == FALSE, scale := 0]  M[, recovered := abundance * scale]  Mtot <- M[, .(sim = sum(simulated), est = sum(recovered)), by = .(method, power, del_prob)]  cat("methods:", paste(sort(unique(M$method)), collapse = ", "), "\n")} else {  cat("abundance_methods.csv not found — run scripts/python/synthetic_abundance.py after the baselines.\n")}

In [ ]:
# --- Figure: abundace_by_insertion_rate (all methods vs 2^p truth) ---------if (have_methods) {  rate <- Mtot[, .(recovered = mean(est)), by = .(method, power)][order(power)]  truth <- unique(Mtot[, .(power)])[order(power)][, truth := 2^power]  fig_rate <- ggplot(rate, aes(factor(power), recovered, fill = method)) +    geom_col(position = position_dodge(0.85), width = 0.8) +    geom_line(data = truth, aes(factor(power), truth, group = 1, colour = "Simulated (2^p)"),              inherit.aes = FALSE, linewidth = 1) +    geom_point(data = truth, aes(factor(power), truth, colour = "Simulated (2^p)"),               inherit.aes = FALSE, size = 2.2) +    scale_x_discrete(labels = function(p) parse(text = paste0("2^", p))) +    scale_y_continuous(trans = "log2") +    scale_fill_manual(values = OKABE_ITO, name = NULL) +    scale_colour_manual(values = c("Simulated (2^p)" = "#000000"), name = NULL) +    labs(x = "Insertion level (log2)", y = "Estimated abundance (log2)") +    theme(legend.position = "bottom")  save_fig(fig_rate, "abundace_by_insertion_rate", 9, 5)  fig_rate}

In [ ]:
# --- Figure: synthetic_methods_r2 (per-method R^2, per-locus vs total) ------if (have_methods) {  r2_locus <- M[, .(level = "per-locus", r2 = r2(simulated, recovered)), by = method]  r2_total <- Mtot[, .(level = "total (family)", r2 = r2(sim, est)), by = method]  r2_methods <- rbindlist(list(r2_locus, r2_total))  r2_methods[, level := factor(level, levels = c("per-locus", "total (family)"))]  fwrite(dcast(r2_methods, method ~ level, value.var = "r2"),         file.path(results_dir, "r2_by_method.csv"))  fig_mr2 <- ggplot(r2_methods[!is.na(r2)], aes(reorder(method, r2), r2, fill = level)) +    geom_col(position = position_dodge(0.8), width = 0.72) +    geom_text(aes(label = sprintf("%.2f", r2)), position = position_dodge(0.8),              vjust = -0.35, size = 3.4) +    scale_fill_manual(values = OKABE_ITO[c(6, 3)], name = "Aggregation") +    coord_cartesian(ylim = c(0, 1.1)) +    labs(x = NULL, y = expression(R^2 ~ "(recovered vs simulated)")) +    theme(legend.position = "bottom", axis.text.x = element_text(angle = 20, hjust = 1))  save_fig(fig_mr2, "synthetic_methods_r2", 8, 5)  fig_mr2}